In [1]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

True

In [2]:
!pip install -q langchain-core==0.3.80 langchain-text-splitters==0.3.11
!pip install -q langchain-community faiss-cpu tiktoken youtube-transcript-api python-dotenv


In [21]:
!pip install langchain PyPDF2


In [24]:
!pip install pypdf

In [25]:
from langchain.document_loaders import PyPDFLoader

pdf_path = "/content/presentation1.pdf"

# Load PDF
loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"Number of pages loaded: {len(docs)}")
print(docs[0].page_content)

Number of pages loaded: 5
(All figure must be insertable from local device)
Slide 0: Title page with information
Slide 1: Problem Statement
 Rising Spam Threat
 Global Impact (2025)
o $16.2B projected mobile sms scam losses [Kaspersky, 2025]
 Limitations of Rule-based Filtering
 Limitations of Black-box Models: high accuracy, zero transparency
 Dual Failure Modes:
o False Positives Block: Legitimate messages like medical alerts, banking OTPs, and
emergency notifications may be blocked.
 71% of users manually check spam messages daily, leading to “filter
fatigue” [Lookout, 2025]
 a 2022 Gartner survey found 41% of IT administrators citing spam filter
mistakes as a top productivity drain
o False Negatives Enable: Spam messages carrying phishing links, malware, or financial
fraud may go undetected.
 Regulatory Compliance (GDPR Article 22): Automated decisions must be explainable, requiring
transparency in any spam detection system.
Slide 2: Objectives
1. Develop a high-precision S

In [29]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [30]:
len(chunks)

9

In [36]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [37]:
from langchain.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding)

In [38]:
vector_store.index_to_docstore_id

{0: '317e6c9e-372a-4e0d-a6bb-f36ccc997461',
 1: 'be1d0d71-eb7e-4ec6-8b34-5fffecbdccf7',
 2: '3d333df1-8d86-488e-971d-6e1be65c9951',
 3: 'bcd57adf-2a8b-4288-9ca3-caf3fe924053',
 4: '842d60a1-7177-4d79-a9d6-b4d35a5c0f1e',
 5: '6ddd5ad8-f140-4032-9b63-3e0251ab13d5',
 6: '291e3dc1-2ce8-4ea9-89d7-b28e7a8364d7',
 7: 'd4c058ce-2c62-4b18-90c4-a661f94163eb',
 8: 'e4b230de-d357-4e53-9901-f9c1ca0412ab'}

In [41]:
vector_store.get_by_ids(['3d333df1-8d86-488e-971d-6e1be65c9951'])

[Document(id='3d333df1-8d86-488e-971d-6e1be65c9951', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-11-04T07:56:03+05:45', 'author': 'Angel Mainali', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-11-04T07:56:03+05:45', 'sourcemodified': "D:20251104075603+05'45'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/presentation1.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2'}, page_content='3.Trained Model produces:\n\uf0b7 Prediction – Spam / Ham\n\uf0b7 LIME Explainer (Local Explanation)\n\uf0b7 SHAP Explainer (Local Explanation)\n\uf0b7 SHAP Global Explanation (Feature Importance)\n3.LIME Explainer + SHAP Explainer (Local Explanation) → combined into → Hybrid Explanation\n(Combination of Local Explanations)\n4.Prediction – Spam / Ham, Hybrid Explanation, and SHAP Global Explanation → displayed on → Web UI\n/ Streamlit Dashboard\n5.Containerization (Docker)\nSlide 4: Dataset and Preprocessing\nDataset:\nThe study lever

In [42]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':2})

In [43]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x79b29c4da630>, search_kwargs={'k': 2})

In [45]:
retriever.invoke("Tell me about datasets")

[Document(id='3d333df1-8d86-488e-971d-6e1be65c9951', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-11-04T07:56:03+05:45', 'author': 'Angel Mainali', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-11-04T07:56:03+05:45', 'sourcemodified': "D:20251104075603+05'45'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/presentation1.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2'}, page_content='3.Trained Model produces:\n\uf0b7 Prediction – Spam / Ham\n\uf0b7 LIME Explainer (Local Explanation)\n\uf0b7 SHAP Explainer (Local Explanation)\n\uf0b7 SHAP Global Explanation (Feature Importance)\n3.LIME Explainer + SHAP Explainer (Local Explanation) → combined into → Hybrid Explanation\n(Combination of Local Explanations)\n4.Prediction – Spam / Ham, Hybrid Explanation, and SHAP Global Explanation → displayed on → Web UI\n/ Streamlit Dashboard\n5.Containerization (Docker)\nSlide 4: Dataset and Preprocessing\nDataset:\nThe study lever

In [46]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)

model = ChatHuggingFace(llm=llm)

In [47]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template=""" you are an expert assistant. Answer only based on the given context. Dont go out of context. If the context is insufficient, just say no.
    {context}
    question: {question}
    """,
    input_variables = ["context", "question"]
)

In [56]:
#question = "Tell me how was the datasets formed"
question = "tell me capital of nepal"
retrieved_docs = retriever.invoke(question)

In [57]:
full_text = " ".join(doc.page_content.strip() for doc in retrieved_docs)


In [58]:
final_prompt = prompt.invoke({"context": full_text, "question": question})

In [59]:
answer = model.invoke(final_prompt)
print(answer.content)

No. The context is about spam detection systems and methodology, not about asking general knowledge questions.


In [62]:
#Building chains

from langchain.schema.runnable import Runnable, RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser



In [63]:
def format_docs(retrieved_docs):
  context_data = " ".join(doc.page_content.strip() for doc in retrieved_docs)
  return context_data


In [64]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [66]:
parallel_chain.invoke("tell me about dataset")

{'context': '3.Trained Model produces:\n\uf0b7 Prediction – Spam / Ham\n\uf0b7 LIME Explainer (Local Explanation)\n\uf0b7 SHAP Explainer (Local Explanation)\n\uf0b7 SHAP Global Explanation (Feature Importance)\n3.LIME Explainer + SHAP Explainer (Local Explanation) → combined into → Hybrid Explanation\n(Combination of Local Explanations)\n4.Prediction – Spam / Ham, Hybrid Explanation, and SHAP Global Explanation → displayed on → Web UI\n/ Streamlit Dashboard\n5.Containerization (Docker)\nSlide 4: Dataset and Preprocessing\nDataset:\nThe study leverages secondary data from the publicly available Kaggle SMS Spam Collection\ncomprising 5,574 labeled messages (86.6% ham, 13.4% spam) [4]. This dataset aggregates\ndiverse sources to ensure ecological validity:\n\uf0b7 425 spam messages from the Grumbletext website\n\uf0b7 3,375 ham messages randomly sampled from the NUS SMS Corpus (NSC)\n\uf0b7 450 ham messages from Caroline Tag’s PhD thesis\n\uf0b7 1,324 messages (1,002 ham, 322 spam) from S

In [67]:
parser = StrOutputParser()

In [68]:
main_chain = parallel_chain | prompt | model | parser

In [69]:
main_chain.invoke("tell me about dataset")

'The study leverages a publicly available Kaggle SMS Spam Collection dataset, comprising 5,574 labeled messages (86.6% ham, 13.4% spam). The dataset aggregates diverse sources to ensure ecological validity:\n\n- 425 spam messages from the Grumbletext website\n- 3,375 ham messages randomly sampled from the NUS SMS Corpus (NSC)\n- 450 ham messages from Caroline Tag’s PhD thesis\n- 1,324 messages (1,002 ham, 322 spam) from SMS Spam Corpus v.0.1 Big'